In [ ]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb

In [ ]:

df = pd.read_csv('data_out/edge_features.csv')

# Target variable creation - binary where 1 indicates at least 1 crime
df['is_dangerous'] = (df['crime_count'] > 0).astype(int)

# === Encode features ===
# Excluding routing identifiers ('edge_id', 'u', 'v', 'key', 'name')
# Excluding 'crime_count' and 'crime_per_km' to prevent data leakage.
features = df[['highway', 'length', 'lamp_count', 'lamp_per_km', 'is_lit']]

# The 'highway' column is categorical (e.g., 'residential', 'footway'), so it requires one-hot encoding.
X = pd.get_dummies(features, columns=['highway'], drop_first=True)
X.columns = [re.sub(r'[\[\]<]', '_', str(col)) for col in X.columns]
y = df['is_dangerous']

# === Train/Test split using stratify (for balance) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# === init XGBoost ===
model = xgb.XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss', 
    random_state=42
)
model.fit(X_train, y_train)

# === validation using AUROC ===
y_pred_proba = model.predict_proba(X_test)[:, 1]
auroc = roc_auc_score(y_test, y_pred_proba)

print(f"XGBoost AUROC Score: {auroc:.4f}")

# === view feature importance ===
importance = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
print(importance.sort_values(by='Importance', ascending=False))

XGBoost AUROC Score: 0.7792
                           Feature  Importance
78             highway_residential    0.203661
74                    highway_path    0.092096
72                 highway_footway    0.079455
0                           length    0.069703
8   highway__'footway', 'service'_    0.056830
..                             ...         ...
59    highway__'trunk', 'service'_    0.000000
77            highway_primary_link    0.000000
71                highway_corridor    0.000000
82                   highway_steps    0.000000
80          highway_secondary_link    0.000000

[87 rows x 2 columns]


c:\Users\shafi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:54:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# ==========================================
# BASELINE
# ==========================================
# Only structural/spatial features, completely ignoring lighting
features_baseline = df[['highway', 'length']]

# encode + clean
X_base = pd.get_dummies(features_baseline, columns=['highway'], drop_first=True)
X_base.columns = [re.sub(r'[\[\]<]', '_', str(col)) for col in X_base.columns]

# train/test split
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.3, random_state=42, stratify=y
)

# training 
model_base = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_base.fit(X_train_base, y_train)

# Calculate AUROC for the Baseline Model
y_pred_base = model_base.predict_proba(X_test_base)[:, 1]
auroc_base = roc_auc_score(y_test, y_pred_base)


# ==========================================
# ADDED LIGHTING
# ==========================================
# Incorporating your novel environmental data (lighting)
features_enhanced = df[['highway', 'length', 'lamp_count', 'lamp_per_km', 'is_lit']]

# encode + clean
X_enh = pd.get_dummies(features_enhanced, columns=['highway'], drop_first=True)
X_enh.columns = [re.sub(r'[\[\]<]', '_', str(col)) for col in X_enh.columns]

# train/test split
X_train_enh, X_test_enh, _, _ = train_test_split(
    X_enh, y, test_size=0.3, random_state=42, stratify=y
)

# training
model_enh = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_enh.fit(X_train_enh, y_train)

# Calculate AUROC
y_pred_enh = model_enh.predict_proba(X_test_enh)[:, 1]
auroc_enh = roc_auc_score(y_test, y_pred_enh)


# ==========================================
# Evaluation & Output
# ==========================================
print("=== MODEL COMPARISON ===")
print(f"Phase 1 (Baseline AUROC): {auroc_base:.4f}")
print(f"Phase 2 (Enhanced AUROC): {auroc_enh:.4f}")

#% improvements
improvement = ((auroc_enh - auroc_base) / auroc_base) * 100
print(f"Performance Shift:        {improvement:+.2f}%\n")

print("=== FEATURE IMPORTANCE (ENHANCED MODEL) ===")
# This proves to the reader exactly how much the model relied on your lighting features
importance_df = pd.DataFrame({
    'Feature': X_enh.columns,
    'Importance Weight': model_enh.feature_importances_
}).sort_values(by='Importance Weight', ascending=False)

print(importance_df.to_string(index=False))

c:\Users\shafi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:33:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\shafi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:33:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== MODEL COMPARISON ===
Phase 1 (Baseline AUROC): 0.7852
Phase 2 (Enhanced AUROC): 0.7792
Performance Shift:        -0.76%

=== FEATURE IMPORTANCE (ENHANCED MODEL) ===
                                               Feature  Importance Weight
                                   highway_residential           0.203661
                                          highway_path           0.092096
                                       highway_footway           0.079455
                                                length           0.069703
                        highway__'footway', 'service'_           0.056830
                                  highway_unclassified           0.055538
                                            lamp_count           0.048800
                                       highway_service           0.038182
                                     highway_secondary           0.037796
                                         highway_track           0.036661
                 